# LLMInspector — Getting Started

A guided tour of the package: **schema → model → metrics → evaluate → synthesize → report**.

The offline cells (schema, adversarial synthesis, reporting shape) run without credentials. The
cells that call an LLM or ragas need a live Azure OpenAI model and the model dependencies.

## 1. Test cases & datasets (schema layer)

In [ ]:
from llminspector import EvaluationDataset, LLMTestCase

dataset = EvaluationDataset(test_cases=[
    LLMTestCase(
        input="What is the capital of France?",
        actual_output="Paris is the capital of France.",
        expected_output="Paris",
        retrieval_context=["The capital of France is Paris."],
    ),
])
# Load from a spreadsheet instead:  EvaluationDataset.from_excel("data.xlsx")
dataset.to_pandas()

## 2. Configure a model (provider layer)

`Settings` replaces the old `config.ini`. Provide exactly one credential: `api_key` **or**
`azure_ad_token_provider`.

In [ ]:
from llminspector import Settings, AzureOpenAIModel

settings = Settings.from_env()          # reads azure_endpoint / api_version / api_key
model = AzureOpenAIModel(settings)      # or AzureOpenAIModel(settings, azure_ad_token_provider=fn)

## 3. Metrics + 4. Evaluate

Metrics are objects built with the model. `evaluate()` drives them per row, skipping any whose
inputs are missing. `answer_correctness` is reported as the weighted `overall_accuracy` blend.

In [ ]:
from llminspector import (
    FaithfulnessMetric, AnswerCorrectnessMetric, SentimentMetric,
    BertScoreMetric, evaluate, reporting,
)

metrics = [
    FaithfulnessMetric(model),
    AnswerCorrectnessMetric(model),
    SentimentMetric(model, target="actual_output"),
    BertScoreMetric(),
]
result = evaluate(dataset, metrics)
reporting.to_dataframe(result)

## 5. Synthesize test data

The adversarial synthesizer runs offline. Alignment (HF-T5) and RAG (ragas) need extra deps.
Every synthesizer returns an `EvaluationDataset` of `Golden`s; extra columns live in
`Golden.metadata`.

In [ ]:
from llminspector import AdversarialSynthesizer

synth = AdversarialSynthesizer.from_excel(
    "../tests/test_sample/test_adversarialdata.xlsx", capability="all", sample_size=5,
)
synth.generate()
synth.to_pandas().head()

## 6. Report

`reporting.to_dataframe` / `to_excel` / `summary` turn a result into a table, a spreadsheet, or
per-metric numeric stats.

In [ ]:
reporting.summary(result)